# 05 — Machine Learning for Sentiment Analysis
**Covers:** Data loading · Training · Cross-validation · Hyperparameters · Statistical significance · Inference · SHAP · Error analysis · Demo

---

In [ ]:
!pip install -q scikit-learn pandas numpy matplotlib seaborn shap scipy datasets

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline
sns.set_theme(style='whitegrid')

## 1. Loading Data

In [ ]:
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer

ds = load_dataset("glue", "sst2", split="train")
df = pd.DataFrame({'text': ds['sentence'], 'label': ds['label']})

# Simple train/test split for this notebook
from sklearn.model_selection import train_test_split
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df['text'].values, df['label'].values,
    test_size=0.20, stratify=df['label'].values, random_state=42
)

# TF-IDF features
tfidf = TfidfVectorizer(max_features=15000, ngram_range=(1,2), sublinear_tf=True)
X_train = tfidf.fit_transform(X_train_raw)
X_test  = tfidf.transform(X_test_raw)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Positive rate — Train: {y_train.mean():.3f}  Test: {y_test.mean():.3f}")

## 2. Training a Baseline Model (Logistic Regression)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score

lr = LogisticRegression(max_iter=1000, C=1.0, solver='saga', random_state=42)
lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)
print("=== Test Set Performance ===")
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

## 3. Cross-Validation

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = cross_validate(
    lr, X_train, y_train,
    cv=cv,
    scoring=['accuracy', 'f1', 'roc_auc'],
    return_train_score=True,
    n_jobs=-1
)

for metric in ['accuracy', 'f1', 'roc_auc']:
    scores = cv_results[f'test_{metric}']
    print(f"{metric:12s}: {scores.mean():.4f} ± {scores.std():.4f}  (folds: {np.round(scores,4)})")

In [ ]:
# Visualise CV scores
metrics = ['accuracy', 'f1', 'roc_auc']
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, m in zip(axes, metrics):
    scores = cv_results[f'test_{m}']
    ax.bar(range(1, 6), scores, color='#1E88E5', alpha=0.8, edgecolor='white')
    ax.axhline(scores.mean(), color='red', linestyle='--', label=f'Mean={scores.mean():.3f}')
    ax.set_title(m.upper(), fontsize=12)
    ax.set_xlabel('Fold')
    ax.set_ylim(0.7, 1.0)
    ax.legend(fontsize=9)
plt.suptitle('5-Fold CV Results — Logistic Regression', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'C': [0.01, 0.1, 1.0, 10.0],
    'penalty': ['l1', 'l2'],
}
grid = GridSearchCV(
    LogisticRegression(max_iter=500, solver='saga', random_state=42),
    param_grid, cv=3, scoring='f1', n_jobs=-1, verbose=1
)
grid.fit(X_train, y_train)

print(f"Best params : {grid.best_params_}")
print(f"Best CV F1  : {grid.best_score_:.4f}")

best_model = grid.best_estimator_
y_pred_best = best_model.predict(X_test)
print(f"Test F1     : {f1_score(y_test, y_pred_best):.4f}")

In [ ]:
# Heatmap of GridSearch results
results_df = pd.DataFrame(grid.cv_results_)
pivot = results_df.pivot_table(index='param_penalty', columns='param_C', values='mean_test_score')

plt.figure(figsize=(8, 3))
sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlGnBu')
plt.title('Grid Search — Mean CV F1 Score')
plt.tight_layout()
plt.show()

## 5. Statistical Significance — McNemar's Test

In [ ]:
from sklearn.naive_bayes import ComplementNB
from scipy.stats import chi2_contingency

# Train a competing model
nb = ComplementNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)

# McNemar's test
b = np.sum((y_pred_best == y_test) & (y_pred_nb != y_test))  # LR correct, NB wrong
c = np.sum((y_pred_best != y_test) & (y_pred_nb == y_test))  # NB correct, LR wrong

contingency = np.array([[0, b], [c, 0]])
chi2 = (abs(b - c) - 1)**2 / (b + c)
from scipy.stats import chi2 as chi2_dist
p_value = 1 - chi2_dist.cdf(chi2, df=1)

print(f"LR (best) Test Accuracy : {accuracy_score(y_test, y_pred_best):.4f}")
print(f"NB Test Accuracy        : {accuracy_score(y_test, y_pred_nb):.4f}")
print(f"\nMcNemar's test: χ² = {chi2:.4f}, p = {p_value:.4f}")
if p_value < 0.05:
    print("→ Statistically significant difference (p < 0.05)")
else:
    print("→ No statistically significant difference (p ≥ 0.05)")

## 6. Inference — Predicting New Samples

In [ ]:
def predict_sentiment(texts, model, vectorizer):
    """Return label and confidence for a list of texts."""
    X = vectorizer.transform(texts)
    labels = model.predict(X)
    probs  = model.predict_proba(X)[:, 1]  # P(Positive)
    return [
        {'text': t, 'label': 'Positive' if l == 1 else 'Negative', 'confidence': f'{p*100:.1f}%'}
        for t, l, p in zip(texts, labels, probs)
    ]

new_texts = [
    "This film is absolutely breathtaking and one of the best I've seen!",
    "A complete waste of time. Boring and poorly written.",
    "It had some good moments but overall felt mediocre.",
]

for result in predict_sentiment(new_texts, best_model, tfidf):
    print(f"[{result['label']:8s} | {result['confidence']:>6}]  {result['text']}")

## 7. SHAP — Explaining Model Predictions

In [ ]:
# SHAP LinearExplainer for Logistic Regression + TF-IDF
explainer = shap.LinearExplainer(best_model, X_train, feature_perturbation='interventional')

# Explain test set (subset for speed)
shap_values = explainer.shap_values(X_test[:200])
print(f"SHAP values shape: {shap_values.shape}")

In [ ]:
# Summary plot — global feature importance
shap.summary_plot(shap_values, X_test[:200],
                  feature_names=tfidf.get_feature_names_out(),
                  max_display=20, show=True)

In [ ]:
# Bar plot — mean absolute SHAP
shap.summary_plot(shap_values, X_test[:200],
                  feature_names=tfidf.get_feature_names_out(),
                  plot_type='bar', max_display=15, show=True)

In [ ]:
# Force plot for a single prediction
idx = 0
print(f"Text: {X_test_raw[idx]}")
print(f"True: {'Positive' if y_test[idx]==1 else 'Negative'}  | Pred: {'Positive' if y_pred_best[idx]==1 else 'Negative'}")
shap.force_plot(
    explainer.expected_value,
    shap_values[idx],
    X_test[idx],
    feature_names=tfidf.get_feature_names_out(),
    matplotlib=True, show=True
)

## 8. Error Analysis

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_best)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Negative', 'Positive'])
fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix — Best LR Model', fontsize=12)
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"TP={tp}  TN={tn}  FP={fp}  FN={fn}")
print(f"Precision={tp/(tp+fp):.4f}  Recall={tp/(tp+fn):.4f}  F1={2*tp/(2*tp+fp+fn):.4f}")

In [ ]:
# Inspect misclassified examples
errors_df = pd.DataFrame({
    'text': X_test_raw,
    'true': y_test,
    'pred': y_pred_best,
    'prob': best_model.predict_proba(X_test)[:,1]
})
errors_df['correct'] = errors_df['true'] == errors_df['pred']
errors_df['true_name'] = errors_df['true'].map({0:'Negative', 1:'Positive'})
errors_df['pred_name'] = errors_df['pred'].map({0:'Negative', 1:'Positive'})

print("=== False Positives (predicted Positive, actually Negative) ===")
fp_df = errors_df[(errors_df['pred']==1) & (errors_df['true']==0)].sort_values('prob', ascending=False)
print(fp_df[['text','prob']].head(5).to_string())

print("\n=== False Negatives (predicted Negative, actually Positive) ===")
fn_df = errors_df[(errors_df['pred']==0) & (errors_df['true']==1)].sort_values('prob', ascending=True)
print(fn_df[['text','prob']].head(5).to_string())

In [ ]:
# Confidence distribution — correct vs incorrect
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, correct, title in zip(axes, [True, False], ['Correct Predictions', 'Errors']):
    subset = errors_df[errors_df['correct'] == correct]
    ax.hist(subset['prob'], bins=30, color='#1E88E5' if correct else '#E53935', alpha=0.8)
    ax.set_title(f'{title} (n={len(subset)})', fontsize=12)
    ax.set_xlabel('P(Positive)')
    ax.set_ylabel('Count')
plt.suptitle('Model Confidence Distribution', fontsize=13)
plt.tight_layout()
plt.show()

## 9. Interactive Demo

In [ ]:
def sentiment_demo(text: str):
    """Quick interactive sentiment classifier."""
    X = tfidf.transform([text])
    label = best_model.predict(X)[0]
    probs = best_model.predict_proba(X)[0]
    sentiment = 'Positive ✅' if label == 1 else 'Negative ❌'
    
    print(f"\nText      : {text}")
    print(f"Prediction: {sentiment}")
    print(f"Confidence: Negative={probs[0]*100:.1f}%  Positive={probs[1]*100:.1f}%")
    
    # Mini bar chart
    fig, ax = plt.subplots(figsize=(5, 1.5))
    ax.barh(['Negative', 'Positive'], probs, color=['#E53935', '#43A047'], alpha=0.85)
    ax.set_xlim(0, 1)
    ax.set_xlabel('Probability')
    ax.set_title('Sentiment Confidence', fontsize=11)
    plt.tight_layout()
    plt.show()

# Try it!
sentiment_demo("The screenplay is brilliant and the acting superb.")
sentiment_demo("I walked out after 20 minutes. Absolutely terrible.")

---
**Next Notebook →** `06_shallow_deep_learning.ipynb`